In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# 1. Load the dataset
print("Loading dataset...")
df = pd.read_csv('../backend/data/greenhouse_crop_yields.csv')

# 2. FEATURE ENGINEERING (Bridging the CSV to the Frontend)
# The model MUST be trained on the exact column names the frontend sends.
# We map the CSV columns to the expected frontend variables:
df['temperature_c'] = df['avg_temperature_C']
df['soil_moisture'] = df['humidity_percent'] # Using humidity % as a proxy for moisture
# Convert Light Intensity (Lux) into a 0-100% Shading metric
df['shading_percent'] = 100 - (df['light_intensity_lux'] / df['light_intensity_lux'].max() * 100)

# 3. Isolate the exact 4 Inputs (Features) and the 1 Output (Target)
X = df[['crop_type', 'shading_percent', 'temperature_c', 'soil_moisture']]
y = df['yield_kg_per_m2']

# 4. Split the data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Create the Preprocessor (Notice 'crop_type' is now lowercase to match the CSV)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['crop_type'])
    ],
    remainder='passthrough' 
)

# 6. Build the Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

# 7. Train the AI!
print("Training the Random Forest model...")
pipeline.fit(X_train, y_train)

# 8. Test the AI on the 20% unseen data
print("\n--- Model Evaluation ---")
predictions = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Mean Absolute Error: {mae:.2f} kg/m²")
print(f"Accuracy (R² Score): {r2:.2f}")

# 9. Export the trained brain
export_path = '../backend/yield_model.joblib'
joblib.dump(pipeline, export_path)
print(f"\nSuccess! Model securely exported to: {export_path}")

Loading dataset...
Training the Random Forest model...

--- Model Evaluation ---
Mean Absolute Error: 2.45 kg/m²
Accuracy (R² Score): 0.60

Success! Model securely exported to: ../backend/yield_model.joblib
